# 08 | Business Value y selección final

## Objetivo del notebook

Este notebook explica cómo se elige el mejor modelo. La selección final no se hace solo con F1, AUC o accuracy, sino con una función de negocio basada en matriz de confusión.

La razón es simple: en este problema los errores no cuestan lo mismo.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: /Users/alexandralozano/dp261-g1-final 2


## 1. Función económica

```text
Valor = TP*2500 + TN*600 + FP*(-900) + FN*(-4500)
```

La corrección clave respecto a versiones anteriores es que FN ahora tiene penalización fuerte. Esto alinea el modelo con el dolor real del negocio: comprar un auto malo creyendo que era bueno.


In [2]:
import pandas as pd
from src.config import REPORTS_DIR

ranking_path = REPORTS_DIR / 'business_model_ranking.csv'
if ranking_path.exists():
    ranking = pd.read_csv(ranking_path)
    display(ranking[['stage','model','business_value','recall','precision','f2','roc_auc','tp','fp','fn','tn']].head(10))
else:
    print('Aún no existe business_model_ranking.csv. Ejecuta: PYTHONPATH=. python scripts/run_all.py')


,stage,model,business_value,recall,precision,f2,roc_auc,tp,fp,fn,tn
0,advanced_sprint4,Bagging_DecisionTree,151900.0,0.468468,0.295455,0.419355,0.714646,52,124,59,665
1,tuned_sprint4,XGBoost_random_search,148900.0,0.495495,0.282051,0.430360,0.718962,55,140,56,649
2,advanced_sprint4,LightGBM_baseline,148400.0,0.423423,0.313333,0.395623,0.717958,47,103,64,686
3,ensembles_sprint4,Voting_soft_LR_RF_LGBM,142900.0,0.468468,0.285714,0.415335,0.728017,52,130,59,659
4,tuned_sprint4,LightGBM_Optuna_TPE,141400.0,0.522523,0.267281,0.438729,0.740223,58,159,53,630
5,advanced_sprint4,XGBoost_baseline,131900.0,0.531532,0.257642,0.438336,0.717604,59,170,52,619
6,tuned_sprint4,RandomForest_random_search,125400.0,0.486486,0.263415,0.416025,0.738328,54,151,57,638
7,ensembles_sprint4,Stacking_RF_LGBM_LR,116400.0,0.675676,0.227273,0.484496,0.732847,75,255,36,534
8,baseline_sprint3,DecisionTree_baseline,115900.0,0.333333,0.321739,0.330948,0.617237,37,78,74,711
9,baseline_sprint3,RandomForest_baseline,89400.0,0.162162,0.720000,0.191898,0.676915,18,7,93,782


## 2. Orden de selección

El ranking se ordena por:

1. Mayor `business_value`.
2. Mayor recall clase 1.
3. Mayor F2.
4. Mayor precision.

Esto evita elegir un modelo que solo mejora accuracy pero deja pasar Bad Buys.


## 3. Por qué threshold 0.5

En Sprint 5 se puede hablar de umbral óptimo, pero la especificación final del proyecto indica usar el threshold estándar `0.5` para la métrica de negocio final. Por eso `run_all.py` compara modelos con threshold fijo de `0.5`.

El dashboard usa el score para mostrar semáforo:

- Rojo: `score >= 0.50`
- Ámbar: `0.30 <= score < 0.50`
- Verde: `score < 0.30`

El semáforo ayuda a que un usuario comercial tome decisiones sin interpretar curvas ROC.


In [3]:
from src.config import BENEFIT_TP, BENEFIT_TN, COST_FP, COST_FN, BUSINESS_THRESHOLD
print(f'Valor = TP*{BENEFIT_TP} + TN*{BENEFIT_TN} + FP*({COST_FP}) + FN*({COST_FN})')
print('Threshold:', BUSINESS_THRESHOLD)


Valor = TP*2500 + TN*600 + FP*(-900) + FN*(-4500)
Threshold: 0.5


## 4. Resultado final

El modelo final se guarda en:

```text
models/final_model.pkl
handoff/model/final_model.pkl
```

La metadata se guarda en:

```text
models/model_metadata.json
handoff/model/model_metadata.json
```

El modelo guardado es un pipeline completo. Por eso en deployment no se deben crear variables manualmente antes de llamar a la API.
